In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
import os
import glob

from sklearn.preprocessing import (
    LabelEncoder,
    OrdinalEncoder,
    StandardScaler,
    MinMaxScaler,
)

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from typing import Dict
from sklearn.model_selection import cross_val_score, KFold
from sklearn.model_selection import train_test_split
from sklearn.model_selection import TimeSeriesSplit
from sklearn.utils.class_weight import compute_class_weight
import warnings


from sklearn.metrics import (
    roc_auc_score, 
    roc_curve, 
    confusion_matrix, 
    precision_score, 
    recall_score, 
    accuracy_score,
    classification_report
)

warnings.filterwarnings('ignore')

# import warnings
# warnings.filterwarnings('ignore')


In [ ]:

# Get all CSV files from the ../../data/machines folder
csv_files = glob.glob('../../data/azure_pm/machines/*.csv')

# Read each CSV file and create dataframes with the same name as the file
for file_path in csv_files:
    # Extract filename without extension
    file_name = os.path.splitext(os.path.basename(file_path))[0]
    
    # Read CSV and assign to variable with the same name as the file
    globals()[file_name] = pd.read_csv(file_path)
    
    print(f"Loaded {file_name}.csv with shape: {globals()[file_name].shape}")

In [ ]:
# machine_1.head()
machine_100.head()

<br> <br>

## Data Encoding and Scaling

In [ ]:
def encode_and_scale_dataframe(df, target_column='failure', categorical_cols=['model'], numerical_cols=['age', 'volt', 'pressure', 'vibration'], scaling_method='standard', exclude_features=None, remove_excluded=False):
  
    # Handle exclude_features parameter
    if exclude_features is None:
        exclude_features = []
    elif isinstance(exclude_features, str):
        exclude_features = [exclude_features]
    
    # Create a copy to avoid modifying original dataframe
    df_processed = df.copy()
    
    print("🔄 Starting encoding and scaling process...")
    print(f"📊 Original dataframe shape: {df_processed.shape}")
    
    if exclude_features:
        print(f"🚫 Excluding features from processing: {exclude_features}")
        if remove_excluded:
            print(f"🗑️  Features will be removed from output dataframe")
    
    # Dictionary to store encoders for later use
    encoders = {}
    
    # 1. ENCODE CATEGORICAL VARIABLES
    print("\n1️⃣ ENCODING CATEGORICAL VARIABLES")
    print("-" * 40)
    
    # Filter out excluded features from categorical columns
    categorical_cols_to_process = [col for col in categorical_cols if col not in exclude_features]
    excluded_categorical = [col for col in categorical_cols if col in exclude_features]
    
    if excluded_categorical:
        print(f"   🚫 Skipping categorical encoding for: {excluded_categorical}")
    
    for col in categorical_cols_to_process:
        if col in df_processed.columns:
            print(f"   Encoding '{col}'...")
            
            # Create and fit label encoder
            le = LabelEncoder()
            df_processed[col] = le.fit_transform(df_processed[col].astype(str))
            
            # Store encoder for future use
            encoders[col] = le
            
            # Display encoding mapping
            unique_values = df[col].unique()
            encoded_values = le.transform(unique_values.astype(str))
            mapping = dict(zip(unique_values, encoded_values))
            
            print(f"   ✅ {col} encoded: {mapping}")
        else:
            print(f"   ⚠️  Column '{col}' not found in dataframe")
    
    # 2. ENCODE TARGET VARIABLE (FAILURE)
    print(f"\n2️⃣ ENCODING TARGET VARIABLE: '{target_column}'")
    print("-" * 40)
    
    if target_column in exclude_features:
        print(f"   🚫 Skipping target variable encoding (excluded): '{target_column}'")
    elif target_column in df_processed.columns:
        print(f"   Encoding '{target_column}'...")
        
        # Check if target is already numeric
        if df_processed[target_column].dtype in ['object', 'category']:
            le_target = LabelEncoder()
            df_processed[target_column] = le_target.fit_transform(df_processed[target_column].astype(str))
            encoders[target_column] = le_target
            
            # Display target encoding mapping
            unique_targets = df[target_column].unique()
            encoded_targets = le_target.transform(unique_targets.astype(str))
            target_mapping = dict(zip(unique_targets, encoded_targets))
            print(f"   ✅ {target_column} encoded: {target_mapping}")
        else:
            print(f"   ℹ️  {target_column} is already numeric")
    else:
        print(f"   ⚠️  Target column '{target_column}' not found")
    
    # 3. SCALE NUMERICAL FEATURES
    print(f"\n3️⃣ SCALING NUMERICAL FEATURES ({scaling_method.upper()})")
    print("-" * 40)
    
    # Filter out excluded features from numerical columns
    numerical_cols_to_process = [col for col in numerical_cols if col not in exclude_features]
    excluded_numerical = [col for col in numerical_cols if col in exclude_features]
    
    if excluded_numerical:
        print(f"   🚫 Skipping scaling for: {excluded_numerical}")
    
    # Filter numerical columns that exist in dataframe
    existing_numerical_cols = [col for col in numerical_cols_to_process if col in df_processed.columns]
    missing_numerical_cols = [col for col in numerical_cols_to_process if col not in df_processed.columns]
    
    if missing_numerical_cols:
        print(f"   ⚠️  Missing columns: {missing_numerical_cols}")
    
    if existing_numerical_cols:
        print(f"   Scaling columns: {existing_numerical_cols}")
        
        # Choose scaler
        if scaling_method.lower() == 'standard':
            scaler = StandardScaler()
            print("   📏 Using StandardScaler (mean=0, std=1)")
        elif scaling_method.lower() == 'minmax':
            scaler = MinMaxScaler()
            print("   📏 Using MinMaxScaler (range 0-1)")
        else:
            scaler = StandardScaler()
            print("   📏 Default: Using StandardScaler")
        
        # Apply scaling
        df_processed[existing_numerical_cols] = scaler.fit_transform(df_processed[existing_numerical_cols])
        
        print("   ✅ Scaling completed")
        
        # Display scaling statistics
        print("\n   📊 SCALING STATISTICS:")
        for col in existing_numerical_cols:
            original_mean = df[col].mean()
            original_std = df[col].std()
            scaled_mean = df_processed[col].mean()
            scaled_std = df_processed[col].std()
            
            print(f"   {col:12} | Original: μ={original_mean:6.2f}, σ={original_std:6.2f} | "
                  f"Scaled: μ={scaled_mean:6.2f}, σ={scaled_std:6.2f}")
    else:
        scaler = None
        print("   ⚠️  No numerical columns to scale")
    
    # 4. REMOVE EXCLUDED FEATURES (if requested)
    if remove_excluded and exclude_features:
        print(f"\n🗑️  REMOVING EXCLUDED FEATURES")
        print("-" * 40)
        features_to_remove = [col for col in exclude_features if col in df_processed.columns]
        if features_to_remove:
            df_processed = df_processed.drop(columns=features_to_remove)
            print(f"   Removed columns: {features_to_remove}")
        else:
            print("   No excluded features found in dataframe")
    
    # 5. SUMMARY
    print(f"\n{'5️⃣' if remove_excluded else '4️⃣'} PROCESSING SUMMARY")
    print("-" * 40)
    print(f"   📊 Final dataframe shape: {df_processed.shape}")
    print(f"   🔤 Categorical columns encoded: {len([col for col in categorical_cols_to_process if col in encoders])}")
    print(f"   🔢 Numerical columns scaled: {len(existing_numerical_cols) if existing_numerical_cols else 0}")
    print(f"   🚫 Features excluded from processing: {len(exclude_features)}")
    if remove_excluded and exclude_features:
        print(f"   🗑️  Features removed from dataframe: {len([col for col in exclude_features if col in df.columns])}")
    print(f"   🎯 Target variable processed: {'Yes' if target_column in df_processed.columns and target_column not in exclude_features else 'No'}")
    
    return df_processed, encoders, scaler

In [ ]:
# Create a dictionary of all machine dataframes
machine_dataframes = {}

# Get a list of variable names first to avoid iteration issues
var_names = list(globals().keys())

for var_name in var_names:
    if var_name.startswith('machine_') and isinstance(globals()[var_name], pd.DataFrame):
        machine_dataframes[var_name] = globals()[var_name]

# Apply encoding and scaling to all machine dataframes
processed_machine_dataframes = {}
encoders_dict = {}
scalers_dict = {}

print(f"Processing {len(machine_dataframes)} machine dataframes...")
print("=" * 60)

for machine_key, machine_df in machine_dataframes.items():
    print(f"\n🔧 Processing {machine_key}...")
    
    # Apply encoding and scaling
    processed_df, encoders, scaler = encode_and_scale_dataframe(
        df=machine_df,
        target_column='failure',
        categorical_cols=['errorID', 'comp'],
        numerical_cols=['volt', 'rotate', 'pressure', 'vibration'],
        scaling_method='minmax',
        exclude_features=['machineID', "age", "model"],
        remove_excluded=True, 
    )

    
    # Store processed dataframe and encoders/scalers
    processed_machine_dataframes[machine_key] = processed_df
    encoders_dict[machine_key] = encoders
    scalers_dict[machine_key] = scaler
    
    print(f"✅ {machine_key} processed successfully - Shape: {processed_df.shape}")

print(f"\n🎉 All {len(processed_machine_dataframes)} machine dataframes processed!")
print(f"📊 Total processed dataframes: {len(processed_machine_dataframes)}")

In [ ]:
processed_machine_dataframes["machine_1"].head()

In [ ]:
processed_machine_dataframes["machine_1"]["errorID"].value_counts()

<br> <br> <br>

### Create lag features

In [ ]:
## Fixed Temporal Feature Engineering with Binary Target

def create_lag_features(df, target_col='failure', prediction_horizon=24):
    """
    Create lag features and BINARY target for time series prediction with NO future data leakage.
    
    Args:
        df: Input dataframe sorted by datetime
        target_col: Column containing failure information
        prediction_horizon: Hours ahead to predict (default: 24 hours)
    
    Returns:
        DataFrame with lag features and proper BINARY target variable
    """
    print(f"🔄 Creating lag features and BINARY target (predicting {prediction_horizon}h ahead)...")
    print(f"Original target distribution: {df[target_col].value_counts().to_dict()}")
    
    # Ensure data is sorted by time
    df_sorted = df.sort_values('datetime').reset_index(drop=True)
    
    # Convert target to BINARY: 0 = no failure, 1 = any failure
    df_sorted['binary_failure'] = (df_sorted[target_col] > 0).astype(int)
    print(f"Binary failure distribution: {df_sorted['binary_failure'].value_counts().to_dict()}")
    
    # Create target: predict failure within next N hours using rolling window
    df_sorted['target'] = df_sorted['binary_failure'].rolling(window=prediction_horizon, min_periods=1).max().shift(-prediction_horizon).fillna(0).astype(int)
    
    # Remove last N rows where we can't predict the future
    df_sorted = df_sorted.iloc[:-prediction_horizon].copy()
    
    # Create lag features for sensor data (only use PAST data)
    sensor_cols = ['volt', 'rotate', 'pressure', 'vibration']
    
    print(f"Creating lag features for sensor columns: {sensor_cols}")
    
    for col in sensor_cols:
        # Lag features (1, 6, 12, 24 hours ago)
        for lag in [1, 6, 12, 24]:
            df_sorted[f'{col}_lag_{lag}h'] = df_sorted[col].shift(lag)
        
        # Rolling statistics (past 24 hours)
        df_sorted[f'{col}_mean_24h'] = df_sorted[col].rolling(window=24, min_periods=1).mean()
        df_sorted[f'{col}_std_24h'] = df_sorted[col].rolling(window=24, min_periods=1).std()
        df_sorted[f'{col}_min_24h'] = df_sorted[col].rolling(window=24, min_periods=1).min()
        df_sorted[f'{col}_max_24h'] = df_sorted[col].rolling(window=24, min_periods=1).max()
        
        # Rolling statistics (past 6 hours)
        df_sorted[f'{col}_mean_6h'] = df_sorted[col].rolling(window=6, min_periods=1).mean()
        df_sorted[f'{col}_std_6h'] = df_sorted[col].rolling(window=6, min_periods=1).std()
    
    # Error and maintenance lag features
    df_sorted['errorID_lag_1h'] = df_sorted['errorID'].shift(1)
    df_sorted['errorID_lag_6h'] = df_sorted['errorID'].shift(6)
    df_sorted['errorID_lag_12h'] = df_sorted['errorID'].shift(12)
    df_sorted['comp_lag_1h'] = df_sorted['comp'].shift(1)
    df_sorted['comp_lag_6h'] = df_sorted['comp'].shift(6)
    df_sorted['comp_lag_12h'] = df_sorted['comp'].shift(12)
    
    # Count features (past events only)
    df_sorted['error_count_6h'] = df_sorted['errorID'].rolling(window=6, min_periods=1).sum()
    df_sorted['error_count_24h'] = df_sorted['errorID'].rolling(window=24, min_periods=1).sum()
    df_sorted['maint_count_6h'] = df_sorted['comp'].rolling(window=6, min_periods=1).sum()
    df_sorted['maint_count_24h'] = df_sorted['comp'].rolling(window=24, min_periods=1).sum()
    
    # Time features
    df_sorted['hour'] = df_sorted['datetime'].dt.hour
    df_sorted['day_of_week'] = df_sorted['datetime'].dt.dayofweek
    df_sorted['is_weekend'] = (df_sorted['datetime'].dt.dayofweek >= 5).astype(int)
    df_sorted['is_working_hours'] = ((df_sorted['hour'] >= 8) & (df_sorted['hour'] <= 17)).astype(int)
    
    # Hours since last maintenance
    maint_mask = df_sorted['comp'] > 0
    if maint_mask.any():
        last_maint_idx = -1
        hours_since_maint = []
        for i, is_maint in enumerate(maint_mask):
            if is_maint:
                last_maint_idx = i
                hours_since_maint.append(0)
            else:
                hours_since_maint.append(i - last_maint_idx if last_maint_idx >= 0 else i)
        df_sorted['hours_since_maint'] = hours_since_maint
    else:
        df_sorted['hours_since_maint'] = range(len(df_sorted))
    
    # Hours since last error
    error_mask = df_sorted['errorID'] > 0
    if error_mask.any():
        last_error_idx = -1
        hours_since_error = []
        for i, is_error in enumerate(error_mask):
            if is_error:
                last_error_idx = i
                hours_since_error.append(0)
            else:
                hours_since_error.append(i - last_error_idx if last_error_idx >= 0 else i)
        df_sorted['hours_since_error'] = hours_since_error
    else:
        df_sorted['hours_since_error'] = range(len(df_sorted))
    
    # Fill NaN values created by lag features
    df_sorted = df_sorted.fillna(0)
    
    print(f"✅ Created lag features. Final shape: {df_sorted.shape}")
    print(f"   BINARY Target distribution: {df_sorted['target'].value_counts().to_dict()}")
    print(f"   Target classes: {sorted(df_sorted['target'].unique())}")
    print(f"   New feature count: {len([col for col in df_sorted.columns if col not in df.columns])}")
    
    return df_sorted


In [ ]:
# Apply lag feature engineering to all processed machine dataframes
lag_machine_dataframes = {}

print(f"Creating lag features for {len(processed_machine_dataframes)} machine dataframes...")
print("=" * 70)

for machine_key, processed_df in processed_machine_dataframes.items():
    print(f"\n🔧 Processing {machine_key}...")
    
    # Get the original machine dataframe to access datetime column
    original_df = machine_dataframes[machine_key].copy()
    
    # Add datetime back to processed dataframe for lag feature creation
    processed_df_with_datetime = processed_df.copy()
    processed_df_with_datetime['datetime'] = original_df['datetime'].values
    
    # Convert datetime column to proper datetime format
    processed_df_with_datetime['datetime'] = pd.to_datetime(processed_df_with_datetime['datetime'])
    
    # Apply lag feature engineering
    df_with_lags = create_lag_features(
        df=processed_df_with_datetime,
        target_col='failure',
        prediction_horizon=24
    )
    
    # Store the result
    lag_machine_dataframes[machine_key] = df_with_lags
    
    print(f"✅ {machine_key} lag features created - Shape: {df_with_lags.shape}")

print(f"\n🎉 All {len(lag_machine_dataframes)} machine dataframes processed with lag features!")
print(f"📊 Total lag dataframes created: {len(lag_machine_dataframes)}")

# Display sample from one machine
sample_machine = 'machine_100'
if sample_machine in lag_machine_dataframes:
    print(f"\n📋 Sample from {sample_machine}:")
    print(f"Columns ({len(lag_machine_dataframes[sample_machine].columns)}): {list(lag_machine_dataframes[sample_machine].columns)}")
    print(f"Shape: {lag_machine_dataframes[sample_machine].shape}")

In [ ]:
lag_machine_dataframes["machine_1"].head()

In [ ]:
lag_machine_dataframes["machine_1"]["failure"].value_counts()